# PhD Program Application Scraper
---

## Goal

This project was inspired by the web scraping lab we completed in class, where we learned the core logic of fetching web pages and extracting structured information from HTML. In that lab we worked in R using the `rvest` package. For this project, I chose to implement the scraper in Python instead.

That decision came after consulting Claude (Anthropic's AI assistant) about the tradeoffs. The short version: Python's `requests` and `BeautifulSoup` libraries are more widely used and better documented for scraping tasks, and Python's `Selenium` and `Playwright` libraries — which handle JavaScript-rendered pages that `rvest` cannot — are far more stable and easier to set up than the R equivalent (`RSelenium`). Since many university department pages render content dynamically with JavaScript, Python was the more practical choice.

I also want to be transparent that Claude helped me write and debug much of the code in this notebook, particularly the more complex parts like the regular expressions and the link-crawling logic. The detailed comments reflects that I used LLM to make sure I understood what each part was actually doing, rather than treating the code as a black box. This is the same approach I took when working with Claude on the cluster pipeline.

---

## What This Notebook Does

I maintain a list of 29 clinical and neuropsychology PhD programs I am considering applying to. Each program has a different application deadline and a different set of required materials, and this information is often scattered across multiple pages of a department website. Manually visiting every page to compile this information is slow and easy to get wrong.

This notebook automates that process. Starting from each program's department URL, the scraper:
1. Fetches the seed page
2. Follows internal links that look like they lead to admissions or requirements pages
3. Extracts deadline dates and required materials from the combined text
4. Flags programs where automated extraction failed, so I can look those up manually
5. Exports everything to an Excel file for final manual review and editing

---

## Libraries
- `requests` — sends HTTP requests to fetch web pages
- `BeautifulSoup` — parses raw HTML into a navigable structure
- `re` — regular expressions for pattern matching (used to find dates)
- `pandas` — organizes results into a table and exports to Excel
- `time` — adds delays between requests
- `urllib` — utilities for working with URLs

---
## 1. Setup and Configuration

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
from urllib.parse import urljoin, urlparse
import warnings
warnings.filterwarnings('ignore')  # suppress SSL certificate warnings from older university sites

# ---------------------------------------------------------------------------
# Configuration constants
# Defining these at the top means I only need to change them in one place
# if I want to adjust behavior (e.g., scrape faster or follow more sub-pages)
# ---------------------------------------------------------------------------

DELAY = 2
# Seconds to wait between HTTP requests.
# This is called "rate limiting" -- it prevents the scraper from hammering
# a server with too many requests too fast, which can get your IP blocked
# or cause problems for the website. 2 seconds is a reasonable default.

TIMEOUT = 10
# Seconds to wait for a server to respond before giving up.
# Some university servers are slow; without a timeout, the scraper
# could hang indefinitely on an unresponsive page.

MAX_SUBPAGES = 3
# Maximum number of internal sub-links to follow per program.
# We do not want to crawl an entire university website -- just the
# admissions-related pages one level deep from the seed URL.

ADMISSION_KEYWORDS = [
    'apply', 'admission', 'application', 'requirement',
    'prospective', 'how-to-apply', 'graduate-admission'
]
# These are the URL fragments we look for when deciding whether to
# follow a link. A link whose href contains any of these words is
# likely to lead to admissions information.

HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/120.0.0.0 Safari/537.36'
    )
}
# When a browser visits a website, it sends a "User-Agent" string that
# identifies itself. Some servers reject requests that do not include one,
# assuming they are bots. Setting this to a realistic browser string
# makes the scraper look like a normal Chrome browser on a Mac.

print('Setup complete.')

Setup complete.


---
## 2. Load the Program List

In [ ]:
df = pd.read_excel('../data/Grad_School_Master_List.xlsx')
# Strip leading/trailing whitespace from URLs.
# A few entries in the Excel file had trailing spaces, which would cause
# the HTTP request to fail with a subtle error that is hard to diagnose.
df['Dept. Website'] = df['Dept. Website'].str.strip()

print(f'Loaded {len(df)} programs')
df.head()

Loaded 15 programs


,University,Location,Area of Study,Dept. Website
0,University of South Alabama,"Mobile, AL","Combined, Clinical-Counseling PhD",https://www.southalabama.edu/colleges/graduate...
1,Fielding University,"Santa Barbara, CA",Clinical Neuropsychology PhD,http://www.fielding.edu/our-programs/school-of...
2,Loma Linda University,"Loma Linda, CA",Clinical Neuropsychology PhD,http://behavioralhealth.llu.edu/programs/psych...
3,Palo Alto University*,"Palo Alto, CA",Clinical Neuropsychology PhD,https://www.paloaltou.edu/graduate-programs/ph...
4,University of Southern California,"Los Angeles, CA",Clinical Neuropsychology (may be with geropsyc...,http://dornsife.usc.edu/psyc/clinical-science/


---
## 3. Core Scraping Functions

The scraper is broken into four focused functions rather than one large block of code. This mirrors the modular approach introduced in the class lab and makes each step easier to test and debug independently.

| Function | What it does |
|---|---|
| `fetch_page(url)` | Fetches one URL, returns a parsed HTML object |
| `find_admission_links(soup, base_url)` | Finds internal links likely leading to admissions info |
| `extract_deadline(text)` | Searches plain text for date patterns using regex |
| `extract_materials(text)` | Searches plain text for application material keywords |
| `scrape_program(seed_url)` | Orchestrates all of the above for one program |

In [3]:
def fetch_page(url):
    """
    Send an HTTP GET request to `url` and return a BeautifulSoup object
    representing the parsed HTML. Returns None if anything goes wrong.
    """
    try:
        response = requests.get(
            url,
            headers=HEADERS,   # send browser-like headers (defined above)
            timeout=TIMEOUT,   # give up after TIMEOUT seconds
            verify=False       # skip SSL certificate verification
                               # (some university sites have expired certificates;
                               #  this prevents a crash, though it is less secure)
        )
        response.raise_for_status()
        # raise_for_status() throws an error if the server returned a
        # failure code like 404 (not found) or 403 (forbidden).
        # Without this, a failed request would silently return an empty page.

        return BeautifulSoup(response.text, 'html.parser')
        # response.text is the raw HTML string.
        # BeautifulSoup parses it into a navigable tree structure
        # so we can search for tags, links, and text.
        # 'html.parser' is Python's built-in HTML parser (no extra install needed).

    except Exception as e:
        print(f'    [FETCH ERROR] {url}: {e}')
        return None
        # Catching all exceptions here ensures one bad URL does not crash
        # the entire loop. The error is printed so we can review it later.


def find_admission_links(soup, base_url):
    """
    Given a parsed page (`soup`) and the URL it came from (`base_url`),
    find internal links that are likely to lead to admissions information.
    Returns a list of absolute URLs (up to MAX_SUBPAGES).
    """
    base_domain = urlparse(base_url).netloc
    # urlparse breaks a URL into components. .netloc gives just the domain,
    # e.g. 'psychology.yale.edu'. We use this to check whether a link
    # stays within the same site.

    found = []

    for tag in soup.find_all('a', href=True):
        # soup.find_all('a', href=True) returns every anchor tag (<a>)
        # that has an href attribute -- i.e., every clickable link on the page.

        href = tag['href'].lower()
        # tag['href'] gets the raw href value, e.g. '/graduate/apply'.
        # .lower() makes the keyword check case-insensitive.

        full_url = urljoin(base_url, tag['href'])
        # urljoin converts a relative URL like '/graduate/apply' into
        # an absolute URL like 'https://psychology.yale.edu/graduate/apply'
        # using the base URL as context. Absolute URLs are left unchanged.

        link_domain = urlparse(full_url).netloc

        # Only keep the link if:
        # (a) it stays on the same domain (internal link), AND
        # (b) its href contains one of our admission keywords
        if link_domain == base_domain:
            if any(kw in href for kw in ADMISSION_KEYWORDS):
                if full_url not in found:  # avoid duplicates
                    found.append(full_url)

    return found[:MAX_SUBPAGES]
    # Slice to MAX_SUBPAGES so we do not follow every matching link --
    # just the first few, which are usually the most relevant.


def extract_deadline(text):
    """
    Search plain text for an application deadline date.
    Uses regular expressions to match common date formats.
    Returns the first match found as a string, or None.
    """

    # --- Step 1: narrow the search area ---
    # Instead of searching the entire page for any date, first look for
    # sentences that mention deadline-related words. This avoids picking up
    # unrelated dates like a professor's publication year.
    deadline_context = re.findall(
        r'(?:deadline|due|closes?|submit by|applications? due)[^.]{0,80}',
        text,
        re.IGNORECASE
        # re.IGNORECASE makes the match work regardless of capitalization.
        # [^.]{0,80} means: capture up to 80 characters that are not a period
        # (i.e., stop at the end of the sentence).
    )
    search_text = ' '.join(deadline_context) if deadline_context else text
    # If deadline-context sentences were found, search only those.
    # If not (rare), fall back to searching the full page text.

    # --- Step 2: define date patterns ---
    patterns = [
        # Pattern 1: written-out month + day + optional year
        # Matches: 'December 1, 2025' / 'Dec 1' / 'November 15th, 2025'
        r'(?:January|February|March|April|May|June|July|August|September|October|November|December'
        r'|Jan|Feb|Mar|Apr|Jun|Jul|Aug|Sep|Oct|Nov|Dec)'
        r'\.?\s+'           # optional period after abbreviated month, then space
        r'\d{1,2}'          # 1 or 2 digit day
        r'(?:st|nd|rd|th)?' # optional ordinal suffix (1st, 2nd, etc.)
        r'(?:,?\s*\d{4})?', # optional comma + 4-digit year

        # Pattern 2: numeric date
        # Matches: '12/01/2025' or '12-01-2025'
        r'\b\d{1,2}[/\-]\d{1,2}[/\-]\d{2,4}\b',
        # \b is a word boundary -- ensures we match a standalone date,
        # not a number embedded inside another string.
    ]

    # --- Step 3: try each pattern, return first match ---
    for pattern in patterns:
        match = re.search(pattern, search_text, re.IGNORECASE)
        if match:
            return match.group(0).strip()
            # match.group(0) returns the full matched string.
            # .strip() removes any accidental leading/trailing whitespace.

    return None  # no date found


def extract_materials(text):
    """
    Search plain text for standard application material keywords.
    Returns a list of materials found (e.g., ['CV / Resume', 'GRE Scores']).
    """

    # Each entry maps a human-readable material name to a list of keywords
    # that indicate it is required. Using a list of synonyms per material
    # handles the variety in how programs describe the same thing.
    materials_map = {
        'Personal Statement / SOP': [
            'personal statement', 'statement of purpose', 'statement of intent'
        ],
        'CV / Resume': [
            'curriculum vitae', r'\bcv\b', 'resume'
            # \bcv\b uses word boundaries to match 'CV' as a standalone word,
            # not as part of words like 'receive' or 'ActiveVolunteer'.
        ],
        'Letters of Recommendation': [
            'letter of recommendation', 'letters of recommendation',
            'recommendation letter', r'\blor\b'
        ],
        'Transcripts': [
            'transcript', 'academic record'
        ],
        'GRE Scores': [
            r'\bgre\b', 'graduate record exam'
        ],
        'Writing Sample': [
            'writing sample'
        ],
        'Research Statement': [
            'research statement', 'research interests'
        ],
        'TOEFL / IELTS': [
            r'\btoefl\b', r'\bielts\b', 'english proficiency'
        ],
    }

    found = []
    text_lower = text.lower()  # lowercase once here so we do not repeat it in every search

    for material, keywords in materials_map.items():
        for kw in keywords:
            if re.search(kw, text_lower):
                found.append(material)
                break
                # break exits the inner loop as soon as one keyword matches.
                # This prevents the same material from being added twice
                # if multiple synonyms appear on the page.

    return found


def scrape_program(seed_url):
    """
    Main function. Scrapes one PhD program starting from `seed_url`.
    Fetches the seed page, follows relevant sub-links, then extracts
    deadline and materials from the combined text.
    Returns a dict with all findings and status metadata.
    """

    # Initialize the result dict with default values.
    # This ensures the dict always has the same keys regardless of
    # what succeeds or fails during scraping.
    result = {
        'deadline':      None,
        'materials':     [],
        'pages_scraped': [],
        'status':        'ok'
    }

    # --- Step 1: fetch the seed page ---
    soup = fetch_page(seed_url)
    if soup is None:
        result['status'] = 'failed - could not fetch seed URL'
        return result  # early return: no point continuing if the main page failed

    result['pages_scraped'].append(seed_url)
    all_text = soup.get_text(separator=' ', strip=True)
    # get_text() extracts all visible text from the HTML, stripping all tags.
    # separator=' ' puts a space between adjacent text blocks so words
    # from different elements do not get concatenated without a gap.

    # --- Step 2: follow admissions sub-links ---
    sub_links = find_admission_links(soup, seed_url)
    for link in sub_links:
        time.sleep(DELAY)  # wait between requests
        sub_soup = fetch_page(link)
        if sub_soup:
            all_text += ' ' + sub_soup.get_text(separator=' ', strip=True)
            # Append the sub-page text to the running body of text.
            # By the end of this loop, all_text contains content from
            # the seed page plus up to MAX_SUBPAGES sub-pages.
            result['pages_scraped'].append(link)

    # --- Step 3: extract information from combined text ---
    result['deadline']  = extract_deadline(all_text)
    result['materials'] = extract_materials(all_text)

    # Update status if deadline was not found (not a failure, just incomplete)
    if not result['deadline']:
        result['status'] = 'ok - deadline not found'

    return result


print('All functions defined.')

All functions defined.


---
## 4. Test on a Single Program

Before running the full list, I test on one program to confirm the scraper works and to see what the output looks like. This is a habit from the class lab: always test on one case before looping over many.

I use the University of Connecticut (row 11) as the test case.

In [4]:
test_row = df.iloc[11]  # .iloc selects a row by its integer position
print(f"Testing: {test_row['University']}")
print(f"URL:     {test_row['Dept. Website']}")
print()

test_result = scrape_program(test_row['Dept. Website'])

print(f"Status:        {test_result['status']}")
print(f"Deadline:      {test_result['deadline']}")
print(f"Materials:     {test_result['materials']}")
print(f"Pages visited: {test_result['pages_scraped']}")

Testing: University of Hawaii
URL:     http://www.psychology.hawaii.edu/concentrations/clinical-psychology.html

Status:        ok - deadline not found
Deadline:      None
Materials:     ['Letters of Recommendation', 'GRE Scores']
Pages visited: ['http://www.psychology.hawaii.edu/concentrations/clinical-psychology.html']


---
## 5. Run on All 29 Programs

Now run the scraper across the full list. Progress is printed in real time.

Three outcomes are expected and all are useful:
- **Full success** — deadline and materials both found
- **Partial** — materials found but deadline missing (or vice versa)
- **Failure** — could not fetch the page at all (blocked, 404, JS-rendered)

Programs in the partial or failure category will be flagged in the next step for manual lookup.

In [5]:
results = []  # will hold one dict per program
n = len(df)

for i, row in df.iterrows():
    # df.iterrows() yields (index, row) pairs, letting us loop through
    # the dataframe one row at a time.

    university = row['University']
    url        = row['Dept. Website']

    print(f'[{i+1:02d}/{n}] {university}')
    # :02d formats the number with a leading zero if needed (e.g., 01, 02 ... 29)
    # so the progress counter stays neatly aligned.

    print(f'        {url}')

    result = scrape_program(url)

    print(f'        Deadline:  {result["deadline"] or "not found"}')
    print(f'        Materials: {len(result["materials"])} found')
    print(f'        Status:    {result["status"]}')
    print()

    # Build a flat dict for this program that merges original data with scraped data.
    # This is what will become one row in the final output dataframe.
    results.append({
        'University':         university,
        'Location':           row['Location'],
        'Area of Study':      row['Area of Study'],
        'Dept. Website':      url,
        'Deadline (scraped)': result['deadline'],
        'Materials Found':    ', '.join(result['materials']) if result['materials'] else None,
        # ', '.join() converts a list like ['CV', 'GRE'] into a single string 'CV, GRE'
        # so it fits cleanly into a spreadsheet cell.
        'Pages Scraped':      len(result['pages_scraped']),
        'Scrape Status':      result['status'],
        'Pages Visited':      ' | '.join(result['pages_scraped'])
        # Store all visited URLs joined by ' | ' for transparency/debugging.
    })

    time.sleep(DELAY)  # pause before the next program

print('Scraping complete.')

[01/15] University of South Alabama
        https://www.southalabama.edu/colleges/graduateschool/ccp/
        Deadline:  March 27, 2026
        Materials: 1 found
        Status:    ok

[02/15] Fielding University
        http://www.fielding.edu/our-programs/school-of-psychology/phd-clinical-psychology/
    [FETCH ERROR] http://www.fielding.edu/admissions/index.php#admissions-advisors: 404 Client Error: Not Found for url: https://www.fielding.edu/admissions/index.php#admissions-advisors
        Deadline:  March 31, 2026
        Materials: 6 found
        Status:    ok

[03/15] Loma Linda University
        http://behavioralhealth.llu.edu/programs/psychology/phd-clinical-psychology
        Deadline:  May 7
        Materials: 2 found
        Status:    ok

[04/15] Palo Alto University*
        https://www.paloaltou.edu/graduate-programs/phd-programs/phd-clinical-psychology/areas-emphasis/neuropsychology
    [FETCH ERROR] https://www.paloaltou.edu/graduate-programs/phd-programs/phd-clinic

---
## 6. Compile Results and Summary

In [6]:
results_df = pd.DataFrame(results)
# pd.DataFrame() converts the list of dicts into a dataframe.
# Each dict becomes one row; the dict keys become column names.

n_total     = len(results_df)
n_deadline  = results_df['Deadline (scraped)'].notna().sum()
# .notna() returns True for cells that are not NaN (i.e., have a value).
# .sum() counts the Trues, giving the total number of programs with a deadline found.

n_materials = results_df['Materials Found'].notna().sum()
n_failed    = results_df['Scrape Status'].str.contains('failed').sum()
# .str.contains('failed') checks whether the status string includes 'failed',
# distinguishing hard failures from programs that loaded but had no deadline.

print('=== Scrape Summary ===')
print(f'Total programs:       {n_total}')
print(f'Deadlines found:      {n_deadline} ({n_deadline/n_total*100:.0f}%)')
print(f'Materials found:      {n_materials} ({n_materials/n_total*100:.0f}%)')
print(f'Hard failures:        {n_failed}')
print(f'Needs manual review:  {n_total - n_deadline}')
print()

results_df[['University', 'Deadline (scraped)', 'Materials Found', 'Scrape Status']]

=== Scrape Summary ===
Total programs:       15
Deadlines found:      7 (47%)
Materials found:      13 (87%)
Hard failures:        1
Needs manual review:  8



,University,Deadline (scraped),Materials Found,Scrape Status
0,University of South Alabama,"March 27, 2026",Transcripts,ok
1,Fielding University,"March 31, 2026","Personal Statement / SOP, CV / Resume, Letters...",ok
2,Loma Linda University,May 7,"Letters of Recommendation, Research Statement",ok
3,Palo Alto University*,NaN,NaN,failed - could not fetch seed URL
4,University of Southern California,March 20,"Personal Statement / SOP, CV / Resume, Researc...",ok
5,University of Colorado at Boudler,November 20,"Personal Statement / SOP, GRE Scores, Research...",ok
6,Yale University,NaN,"Letters of Recommendation, Transcripts, GRE Sc...",ok - deadline not found
7,Catholic University of America,NaN,"Personal Statement / SOP, CV / Resume, Letters...",ok - deadline not found
8,Nova Southeastern University,NaN,"Letters of Recommendation, Transcripts",ok - deadline not found
9,University of Georgia,NOVEMBER 15,"Personal Statement / SOP, CV / Resume, Letters...",ok


---
## 7. Flag Programs Needing Manual Review

Programs where the deadline could not be found automatically are listed here. The most common reasons:

- The page uses JavaScript to load content dynamically. `requests` fetches the raw HTML before JavaScript runs, so the deadline never appears in the text.
- The deadline is only listed in a PDF linked from the page, which this scraper does not open.
- The URL in the original list is outdated and returned a 404 or redirect.

These are real limitations of a `requests` + `BeautifulSoup` scraper. Handling JavaScript pages would require `Playwright` or `Selenium` — a possible extension for a future version of this project.

In [7]:
needs_review = results_df[results_df['Deadline (scraped)'].isna()].copy()
# Boolean indexing: selects only rows where the deadline column is NaN.
# .copy() creates an independent copy so edits do not affect results_df.

print(f'{len(needs_review)} programs flagged for manual deadline lookup:\n')
for _, row in needs_review.iterrows():
    print(f"  {row['University']}")
    print(f"  {row['Dept. Website']}")
    print(f"  Status: {row['Scrape Status']}")
    print()

8 programs flagged for manual deadline lookup:

  Palo Alto University*
  https://www.paloaltou.edu/graduate-programs/phd-programs/phd-clinical-psychology/areas-emphasis/neuropsychology
  Status: failed - could not fetch seed URL

  Yale University
  http://psychology.yale.edu/graduate/training/programs-study
  Status: ok - deadline not found

  Catholic University of America
  https://psychology.catholic.edu/academics/graduate/phd-clinical-psychology/index.html
  Status: ok - deadline not found

  Nova Southeastern University
  http://psychology.nova.edu/graduate/clinical-psychology/tracks.html
  Status: ok - deadline not found

  Georgia State University
  http://psychology.gsu.edu/graduate/program-areas-and-concentrations/clinical-program/clinical-neuropsychology/
  Status: ok - deadline not found

  University of Hawaii
  http://www.psychology.hawaii.edu/concentrations/clinical-psychology.html
  Status: ok - deadline not found

  Northwestern University Feinberg School of Medicine


---
## 8. Export to Excel

Export the full results table to Excel. Programs where the deadline was not found will have an empty cell in the `Application Deadline` column — I will fill those in manually after visiting each flagged program's website.

In [ ]:
export_df = results_df.rename(columns={
    'Deadline (scraped)': 'Application Deadline'
    # Rename for clarity in the final file -- 'scraped' is an implementation
    # detail that does not need to appear in the exported spreadsheet.
})

export_df = export_df.sort_values('Application Deadline', na_position='last')
# Sort so programs with found deadlines appear first, empty ones at the bottom.
# na_position='last' ensures NaN values (missing deadlines) go to the end.

export_df.to_excel('../data/phd_programs_results.xlsx', index=False)
# index=False prevents pandas from writing the row numbers as an extra column.

print('Exported: phd_programs_results.xlsx')
print(f'  {n_deadline} deadlines filled in automatically')
print(f'  {n_total - n_deadline} rows left blank for manual entry')

export_df[['University', 'Application Deadline', 'Materials Found', 'Scrape Status']].head(10)

Exported: phd_programs_results.xlsx
  7 deadlines filled in automatically
  8 rows left blank for manual entry


,University,Application Deadline,Materials Found,Scrape Status
12,Loyola University Chicago,July 20,NaN,ok
4,University of Southern California,March 20,"Personal Statement / SOP, CV / Resume, Researc...",ok
0,University of South Alabama,"March 27, 2026",Transcripts,ok
1,Fielding University,"March 31, 2026","Personal Statement / SOP, CV / Resume, Letters...",ok
2,Loma Linda University,May 7,"Letters of Recommendation, Research Statement",ok
9,University of Georgia,NOVEMBER 15,"Personal Statement / SOP, CV / Resume, Letters...",ok
5,University of Colorado at Boudler,November 20,"Personal Statement / SOP, GRE Scores, Research...",ok
3,Palo Alto University*,NaN,NaN,failed - could not fetch seed URL
6,Yale University,NaN,"Letters of Recommendation, Transcripts, GRE Sc...",ok - deadline not found
7,Catholic University of America,NaN,"Personal Statement / SOP, CV / Resume, Letters...",ok - deadline not found
